# De la Emoción a la Ecuación · Viernes 20
## Captura y Alfa · velas de una hora y de un día

**Castle Quant · La ciencia del alfa.** Método CASTLE: hoy **C · Captura** y **A · Alfa**.

El modelo es una regresión lineal simple sobre los retornos, un AR(1), con un peso $w$ y un sesgo $b$. Se entrena y se prueba dos veces: con velas de una hora y con velas de un día.

$$r_t = b + w \cdot r_{t-1} + \varepsilon_t$$

| # | Sección | Qué sale |
|---|---|---|
| 1 | Preparación | Las librerías y el estilo de las gráficas |
| 2 | Parámetros | El activo y, por frecuencia, el tamaño de la vela, la historia y la fecha de corte |
| 3 | Captura | `crudas`: las velas de una hora y las de un día, tal como llegan |
| 4 | Partición | `entrenamiento_crudo` y `prueba_cruda`: cada frecuencia partida por su fecha de corte |
| 5 | Limpieza | `entrenamiento` y `prueba`: cada tramo limpio por separado |
| 6 | Retornos | `retornos`: el retorno logarítmico de cada vela del entrenamiento |
| 7 | Variables | `pares`: el retorno de la vela anterior ($x$) junto al retorno de la vela ($y$) |
| 8 | Entrenamiento | `modelo`: el peso y el sesgo de cada frecuencia |
| 9 | Ajuste | `ajuste`: cuánto explica el modelo en el entrenamiento |
| 10 | Prueba | `comparacion`: el mismo modelo medido en la prueba |
| 11 | Predicción | El retorno que se espera en la próxima vela y la señal |
| 12 | Reto de la semana | El mismo notebook con otro activo |
| 13 | Guardar | Los archivos para la próxima sesión |

> Contenido educativo; no es recomendación de inversión. Los resultados históricos no garantizan resultados futuros.

## 1 · Preparación

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

# Colores de Castle Quant para las gráficas
NAVY, LINEA, TEXTO, GRIS, AZUL, DORADO = "#0b1a2e", "#22395a", "#f5f7fa", "#8e9ab5", "#7fa3e6", "#cfb97d"
plt.rcParams.update({
    "figure.facecolor": NAVY, "axes.facecolor": NAVY, "axes.edgecolor": LINEA, "axes.labelcolor": TEXTO,
    "text.color": TEXTO, "xtick.color": GRIS, "ytick.color": GRIS, "grid.color": LINEA, "axes.grid": True,
    "axes.titleweight": "bold", "figure.figsize": (12, 5), "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False,
})

## 2 · Parámetros

La única celda que se cambia. El peso y el sesgo no están aquí: los estima el modelo en la sección 8.

In [ ]:
ACTIVO = "BTC-USD"     # Bitcoin contra dólar en Yahoo Finance

# Por frecuencia: el tamaño de la vela, cuánta historia se pide y desde qué día todo es prueba
FRECUENCIAS = {
    "hora": {"intervalo": "1h", "periodo": "730d", "corte": "2026-04-01"},     # Yahoo entrega velas de una hora de los últimos 730 días
    "dia":  {"intervalo": "1d", "periodo": "max",  "corte": "2023-01-01"},     # velas de un día desde 2014
}

---
# C · Captura

## 3 · Captura

**Entra:** `ACTIVO`, `FRECUENCIAS` · **Sale:** `crudas`, con las velas de cada frecuencia tal como llegan

In [ ]:
def capturar(activo, intervalo, periodo):
    """Descarga las velas de un activo: una fila por vela con apertura, máximo, mínimo y cierre."""
    datos = yf.download(activo, period=periodo, interval=intervalo, auto_adjust=True, progress=False)
    datos.columns = datos.columns.get_level_values(0)     # yfinance entrega columnas en dos niveles: se deja uno

    velas = datos.rename(columns={"Open": "apertura", "High": "maximo", "Low": "minimo", "Close": "cierre"})
    velas = velas[["apertura", "maximo", "minimo", "cierre"]].rename_axis(columns=None)
    velas.index = velas.index.tz_localize(None).rename("Fecha")     # las fechas quedan en UTC
    return velas


crudas = {nombre: capturar(ACTIVO, f["intervalo"], f["periodo"]) for nombre, f in FRECUENCIAS.items()}

for nombre, velas in crudas.items():
    print(f"{nombre:<5} {len(velas):>6,} velas · del {velas.index[0]:%Y-%m-%d %H:%M} al {velas.index[-1]:%Y-%m-%d %H:%M} UTC")

## 4 · Partición

**Entra:** `crudas`, el corte de cada frecuencia · **Sale:** `entrenamiento_crudo` y `prueba_cruda`

La partición va antes de la limpieza. Desde aquí cada tramo se trata por separado, y nada de lo que se haga con el entrenamiento toca la prueba.

In [ ]:
def partir(velas, corte):
    """Parte las velas por fecha: entrenamiento antes del corte, prueba desde el corte."""
    corte = pd.Timestamp(corte)
    return velas[velas.index < corte], velas[velas.index >= corte]


entrenamiento_crudo, prueba_cruda = {}, {}
for nombre, f in FRECUENCIAS.items():
    entrenamiento_crudo[nombre], prueba_cruda[nombre] = partir(crudas[nombre], f["corte"])
    print(f"{nombre:<5} corte {f['corte']} · entrenamiento: {len(entrenamiento_crudo[nombre]):>6,} velas "
          f"({len(entrenamiento_crudo[nombre]) / len(crudas[nombre]):.0%}) · prueba: {len(prueba_cruda[nombre]):>6,} velas")

## 5 · Limpieza

**Entra:** `entrenamiento_crudo`, `prueba_cruda` · **Sale:** `entrenamiento` y `prueba`, sin duplicados, sin precios vacíos y sin la vela en curso

La misma función limpia cada tramo por separado.

In [ ]:
def limpiar(velas, intervalo):
    """Deja una sola vela por fecha, con precios válidos y ya cerrada."""
    ahora = pd.Timestamp.now(tz="UTC").tz_localize(None)

    velas = velas.sort_index()
    velas = velas[~velas.index.duplicated(keep="last")]              # una sola vela por fecha
    velas = velas[(velas > 0).all(axis=1)]                           # sin precios vacíos ni negativos
    return velas[velas.index + pd.Timedelta(intervalo) <= ahora]     # la vela en curso aún no cierra


entrenamiento = {nombre: limpiar(entrenamiento_crudo[nombre], f["intervalo"]) for nombre, f in FRECUENCIAS.items()}
prueba = {nombre: limpiar(prueba_cruda[nombre], f["intervalo"]) for nombre, f in FRECUENCIAS.items()}

for nombre in FRECUENCIAS:
    print(f"{nombre:<5} entrenamiento: {len(entrenamiento_crudo[nombre]):>6,} → {len(entrenamiento[nombre]):>6,} velas · "
          f"prueba: {len(prueba_cruda[nombre]):>6,} → {len(prueba[nombre]):>6,} velas")

In [ ]:
# El entrenamiento de cada frecuencia; la prueba queda como una franja vacía hasta la sección 10
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))
for eje, (nombre, f) in zip(ejes, FRECUENCIAS.items()):
    eje.plot(entrenamiento[nombre]["cierre"], color=AZUL, lw=1.1, label="Entrenamiento")
    eje.axvspan(prueba[nombre].index[0], prueba[nombre].index[-1], color=GRIS, alpha=0.18, label="Prueba")
    eje.set_title(f"{ACTIVO} · cierre · velas de {f['intervalo']}")
    eje.tick_params(axis="x", rotation=30)
    eje.legend(loc="upper left")
plt.show()

## 6 · Retornos

**Entra:** `entrenamiento` · **Sale:** `retornos`, el retorno logarítmico de cada vela del entrenamiento

$$r_t = \ln\left(\frac{C_t}{C_{t-1}}\right)$$

In [ ]:
def retornos_log(velas):
    """Devuelve el retorno logarítmico de cada vela contra la anterior."""
    return np.log(velas["cierre"]).diff().dropna()


retornos = {nombre: retornos_log(entrenamiento[nombre]) for nombre in FRECUENCIAS}

pd.DataFrame({
    nombre: {
        "Retornos": f"{len(r):,}",
        "Retorno logarítmico medio": f"{r.mean():.4%}",
        "Desviación estándar": f"{r.std():.3%}",
        "Velas positivas": f"{(r > 0).mean():.1%}",
    }
    for nombre, r in retornos.items()
})

---
# A · Alfa

## 7 · Variables

**Entra:** `retornos` · **Sale:** `pares`, con `x`, el retorno de la vela anterior, y `y`, el retorno de la vela

$$x_t = r_{t-1} \qquad\qquad y_t = r_t$$

En cada fila, `x` es el `y` de la fila de arriba: el modelo solo usa lo que ya pasó.

In [ ]:
def armar_pares(retornos):
    """Pone junto a cada retorno (y) el retorno de la vela anterior (x)."""
    return pd.DataFrame({
        "x": retornos.shift(1),     # r(t-1): el retorno de la vela anterior, que ya se conoce
        "y": retornos,              # r(t): el retorno que se quiere explicar
    }).dropna()


pares = {nombre: armar_pares(retornos[nombre]) for nombre in FRECUENCIAS}

pares["dia"].tail()

## 8 · Entrenamiento

**Entra:** `pares` · **Sale:** `modelo`, con el peso `w` y el sesgo `b` de cada frecuencia

$$r_t = b + w \cdot r_{t-1} + \varepsilon_t$$

$$w = \frac{\sum (x_t - \bar{x})(y_t - \bar{y})}{\sum (x_t - \bar{x})^2} \qquad\qquad b = \bar{y} - w \cdot \bar{x}$$

De todas las rectas posibles, estas dos fórmulas dan la que deja el menor error al cuadrado.

**Reto en vivo:** antes de ejecutar la celda, ¿el peso $w$ sale positivo, negativo o cero? ¿Sale igual con velas de una hora que con velas de un día? **Responde en tu cuenta gratis de Castle Quant.**

In [ ]:
def estimar(pares):
    """Devuelve el peso w y el sesgo b de la recta y = b + w·x que deja el menor error al cuadrado."""
    x, y = pares["x"], pares["y"]
    w = ((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean()) ** 2).sum()
    b = y.mean() - w * x.mean()
    return pd.Series({"w": w, "b": b})


modelo = pd.DataFrame({nombre: estimar(pares[nombre]) for nombre in FRECUENCIAS})

modelo

## 9 · Ajuste

**Entra:** `pares`, `modelo` · **Sale:** `ajuste`, cuánto explica el modelo en el entrenamiento

$$\hat{y}_t = b + w \cdot x_t \qquad\qquad R^2 = 1 - \frac{\sum (y_t - \hat{y}_t)^2}{\sum (y_t - \bar{y})^2}$$

El acierto de dirección es la fracción de velas en las que $\hat{y}_t$ y $y_t$ tienen el mismo signo.

In [ ]:
def medir(pares, parametros):
    """Mide un modelo sobre unos pares: el R² y la fracción de velas en que acierta la dirección."""
    estimado = parametros["b"] + parametros["w"] * pares["x"]     # lo que la recta dice de cada vela
    error = pares["y"] - estimado                                 # lo que la recta no explica
    return pd.Series({
        "R²": 1 - (error ** 2).sum() / ((pares["y"] - pares["y"].mean()) ** 2).sum(),
        "Acierto de dirección": (np.sign(estimado) == np.sign(pares["y"])).mean(),
    })


ajuste = pd.DataFrame({nombre: medir(pares[nombre], modelo[nombre]) for nombre in FRECUENCIAS})

ajuste.apply(lambda columna: columna.map("{:.3%}".format))

In [ ]:
# La nube de pares (x, y) de cada frecuencia y su recta estimada
fig, ejes = plt.subplots(1, 2, figsize=(14, 7))
for eje, (nombre, f) in zip(ejes, FRECUENCIAS.items()):
    w, b = modelo.loc["w", nombre], modelo.loc["b", nombre]
    limite = pares[nombre].abs().quantile(0.999).max()
    recta_x = np.array([-limite, limite])

    eje.scatter(pares[nombre]["x"], pares[nombre]["y"], s=4, color=AZUL, alpha=0.35)
    eje.plot(recta_x, b + w * recta_x, color=DORADO, lw=2.5, label=f"w = {w:+.4f} · b = {b:+.6f}")
    eje.set_xlim(-limite, limite)
    eje.set_ylim(-limite, limite)
    eje.set_xlabel("x · retorno de la vela anterior")
    eje.set_ylabel("y · retorno de la vela")
    eje.set_title(f"Velas de {f['intervalo']} · cada retorno contra el anterior")
    eje.legend(loc="upper left")
plt.show()

## 10 · Prueba

**Entra:** `prueba`, `modelo` · **Sale:** `comparacion`, el mismo modelo medido en el entrenamiento y en la prueba

El peso y el sesgo no se vuelven a estimar: son los de la sección 8.

In [ ]:
pares_prueba = {nombre: armar_pares(retornos_log(prueba[nombre])) for nombre in FRECUENCIAS}
ajuste_prueba = pd.DataFrame({nombre: medir(pares_prueba[nombre], modelo[nombre]) for nombre in FRECUENCIAS})

comparacion = pd.concat({"entrenamiento": ajuste, "prueba": ajuste_prueba}, axis=1)

for nombre in FRECUENCIAS:
    print(f"{nombre:<5} pares de entrenamiento: {len(pares[nombre]):>6,} · pares de prueba: {len(pares_prueba[nombre]):>6,}")
comparacion.apply(lambda columna: columna.map("{:.3%}".format))

## 11 · Predicción

**Entra:** `prueba`, `modelo` · **Sale:** el retorno que el modelo espera en la próxima vela y la señal

$$\hat{r}_{t+1} = b + w \cdot r_t \qquad\qquad s_t = \begin{cases} \text{largo} & \text{si } \hat{r}_{t+1} > 0\\ \text{corto} & \text{si } \hat{r}_{t+1} \le 0\end{cases}$$

In [ ]:
def predecir(velas, parametros):
    """Con el retorno de la última vela cerrada, calcula el retorno que el modelo espera en la siguiente."""
    retorno = retornos_log(velas).iloc[-1]
    esperado = parametros["b"] + parametros["w"] * retorno
    return pd.Series({
        "Última vela cerrada": f"{velas.index[-1]:%Y-%m-%d %H:%M}",
        "Su retorno": f"{retorno:+.3%}",
        "Retorno esperado en la siguiente": f"{esperado:+.4%}",
        "Señal": "largo" if esperado > 0 else "corto",
    })


pd.DataFrame({nombre: predecir(prueba[nombre], modelo[nombre]) for nombre in FRECUENCIAS})

---
## 12 · Reto de la semana

Cambia `ACTIVO` a `"ETH-USD"` en la sección 2 y ejecuta todo de nuevo. ¿Cambia el signo de $w$ en alguna de las dos frecuencias? ¿El modelo explica más en la prueba? Sube tu captura a la comunidad de Castle Quant.

## 13 · Guardar

**Entra:** `entrenamiento`, `prueba`, `modelo` · **Sale:** por frecuencia, un archivo de entrenamiento y uno de prueba; y un archivo con el peso y el sesgo

In [ ]:
prefijo = ACTIVO.split("-")[0]     # BTC

for nombre in FRECUENCIAS:
    entrenamiento[nombre].to_csv(f"{prefijo}_{nombre}_entrenamiento.csv")
    prueba[nombre].to_csv(f"{prefijo}_{nombre}_prueba.csv")
    print(f"Guardado: {prefijo}_{nombre}_entrenamiento.csv ({len(entrenamiento[nombre]):,} velas) · "
          f"{prefijo}_{nombre}_prueba.csv ({len(prueba[nombre]):,} velas)")

modelo.to_json(f"{prefijo}_modelo.json")
print(f"Guardado: {prefijo}_modelo.json")

---
**La próxima sesión:** Simulación, Tamaño y Libre de sesgos. Lleva los archivos.

Comunidad Castle Quant en WhatsApp: https://chat.whatsapp.com/LfKXZoL6xjRF32c6ZMBKzc